# Week 3 (scaled up): Full-Corpus Financial Sentiment Analysis

**Owner:** Fatima Nasir · **Project:** Market Stress Early-Warning System · **Focus:** NLP / financial sentiment

**Update 2026-11-21.** The first version of this notebook scored 20 filings from 5 technology companies in 2022. One year cannot be tested against our 13 stress episodes (2005 to 2026), so the same steps now run on **50 large S&P 500 companies across all 11 sectors, every 10-K and 10-Q from 2006 to today**. The rules are Fatima's; the code moved into `src/sentiment.py` so `scripts/build_sentiment.py` and the data refresh can run it too. The 2022 version is in the git history.

Goals (unchanged):
1. Score MD&A and Risk Factors paragraphs with FinBERT (Araci, 2019).
2. Analyze Risk Factors and MD&A separately.
3. Measure tone changes between filings.
4. Align filings with trading days, with no look-ahead.
5. Save daily sentiment features, now with a market-wide index for the lead-time tests (notebooks 14 and 15).

**Run order:** `python scripts/build_sentiment.py` downloads the filings and scores them (several hours the first time; later runs only add new filings). This notebook then reuses the saved scores, so it runs in minutes.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CODE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sentiment.py").exists())
DATA_ROOT = Path(os.environ.get("RGBFS_DATA_ROOT", CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT))
from src import sentiment as se

PROC = DATA_ROOT / "data" / "processed"
SEC_DIR, FEATURES = PROC / "sec", PROC / "features"
print(f"{len(se.FIRMS)} firms, from {se.START}")

## 1. Load the SEC filing corpus

Built by `scripts/build_sentiment.py` with `src/sec_filings.py` (company CIK in every URL, older submission files included, inline-XBRL header stripped, section headings found inside small tables). Two firms changed their SEC registrant: Google Inc. (before Alphabet, 2015) and the pre-2019 Walt Disney Company, so their older CIKs are read too. The coverage table shows filings per firm and year: a zero marks a gap to check.

In [ ]:
para_path = SEC_DIR / "paragraphs_full.parquet"
if not para_path.exists():
    raise SystemExit("Run `python scripts/build_sentiment.py` first (it downloads and scores the filings).")
paras = pd.read_parquet(para_path)
paras = paras[paras["ticker"].isin(se.FIRMS)]
filings = pd.read_csv(SEC_DIR / "filings_full.csv")
cov = se.coverage(filings[filings["ticker"].isin(se.FIRMS)])
print(f"{paras['accession'].nunique():,} filings, {len(paras):,} paragraphs, {paras['ticker'].nunique()} firms")
gaps = (cov == 0).sum(axis=1)
print("Firm-years with no filing:", int(gaps.sum()), "| firms with any gap:", ", ".join(gaps[gaps > 0].index) or "none")
cov.sum().to_frame("filings").T

## 2. Prepare text for scoring

As in Week 2 and 3: paragraphs of at least 15 words; MD&A = Item 7 of a 10-K or Item 2 of a 10-Q; Risk Factors = Item 1A.

In [ ]:
sel = se.select_sections(paras)
sel["year"] = pd.to_datetime(sel["filing_date"]).dt.year
print("Paragraphs ready:", f"{len(sel):,}")
pd.crosstab(sel["year"], sel["section_group"]).T

### Filtering repeated Risk Factors

Risk Factors paragraphs that repeat the company's earlier filings (exact or at least 90% similar) are dropped, so boilerplate does not drive the tone. **Change for 20 years of text:** each paragraph is compared with the company's previous four filings (about a year) instead of every earlier filing, with a fast pre-filter on shared three-word phrases. On a realistic test this gives the same result as the full comparison while taking minutes instead of hours. The filter runs before scoring, so FinBERT does not score text that would be dropped. Filings with no new Risk Factors keep missing tone (NaN), not zero.

In [ ]:
sel["is_new"] = se.mark_new_risk_factors(sel)
rf = sel[sel["section_group"].eq("Risk Factors")]
print(f"Risk Factors kept: {int(rf['is_new'].sum()):,} of {len(rf):,} ({rf['is_new'].mean():.0%}); MD&A kept: {int(sel['section_group'].eq('MD&A').sum()):,}")

## 3. FinBERT scores

`ProsusAI/finbert`, one label per paragraph (negative -1, neutral 0, positive +1). Scores are cached by text, so this cell reuses the scores from `scripts/build_sentiment.py` and only scores paragraphs not seen before.

In [ ]:
keep = sel[sel["is_new"]]
scores = se.score_texts(keep["clean_text"], se.finbert_scorer(), cache_path=SEC_DIR / "finbert_cache.parquet")
sel = sel.join(scores[["sentiment", "confidence"]], how="left")
pd.crosstab(sel.loc[sel["is_new"], "section_group"], sel.loc[sel["is_new"], "sentiment"], normalize="index").round(3)

## 4. Filing-level tone and tone changes

Average score of the kept paragraphs per company, filing and section, plus the share of negative paragraphs. Tone change compares a filing with the previous filing of the same form (10-K with 10-K, 10-Q with 10-Q).

In [ ]:
summ = se.align_to_trading_days(se.filing_summary(sel), pd.read_parquet(PROC / "daily_panel.parquet").index)
print("Filing-section records:", len(summ), "| missing tone (no new paragraphs):", int(summ["average_sentiment"].isna().sum()))
summ.groupby("section_group")[["average_sentiment", "tone_change", "negative_share"]].describe().round(3).T

## 5. Daily features with no look-ahead

Each filing is used from the first trading day on or after its filing date and then delayed one more trading day (Fatima's Steps 8 and 11). The market-wide index averages every firm's latest filing; a firm drops out once its latest filing is more than about two quarters old.

In [ ]:
days = pd.read_parquet(PROC / "daily_panel.parquet").index
daily = se.daily_features(summ, days)
market = se.market_daily(daily)
FEATURES.mkdir(parents=True, exist_ok=True)
sel.drop(columns=["text"]).to_parquet(FEATURES / "sec_sentiment_scored.parquet", index=False)
summ.to_parquet(FEATURES / "sec_sentiment_filings.parquet", index=False)
daily.to_parquet(FEATURES / "sec_sentiment_daily.parquet", index=False)
market.to_parquet(FEATURES / "sec_sentiment_market_daily.parquet")
first = daily.dropna(subset=["average_sentiment"]).sort_values("date").groupby(["ticker", "section_group"]).first()
assert (first["date"] > first["source_filing_trading_date"]).all(), "a filing was used on or before its trading day"
print(f"daily rows: {len(daily):,}; market index {market.dropna(how='all').index.min():%Y-%m-%d} to {market.index.max():%Y-%m-%d}")

## 6. Market-wide tone around stress episodes

A first look before the formal lead-time test (notebook 15): does filing tone fall before episodes start, or only once they are under way?

In [ ]:
ep = pd.read_csv(DATA_ROOT / "docs" / "stress_periods.csv", parse_dates=["start", "end"])
m = market.dropna(how="all")
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, col, label in [(axes[0], "mdna_sentiment", "MD&A tone"), (axes[1], "risk_negative_share", "Risk Factors: share negative (new paragraphs)")]:
    ax.plot(m.index, m[col], lw=1)
    for r in ep.itertuples():
        ax.axvspan(r.start, r.end, color="tab:red", alpha=0.12, lw=0)
    ax.set_ylabel(label)
axes[0].set_title("Market-wide filing tone, 50 firms (red = labeled stress episodes)")
plt.tight_layout(); plt.show()

stress = pd.read_parquet(PROC / "stress_labels_daily.parquet")["stress"].reindex(m.index)
pre = pd.Series(False, index=m.index)
for r in ep.itertuples():
    pre |= (m.index >= r.start - pd.tseries.offsets.BDay(60)) & (m.index < r.start)
groups = pd.Series(np.select([stress.eq(1), pre], ["in episode", "60 days before a start"], "other days"), index=m.index)
m[["mdna_sentiment", "mdna_tone_change", "risk_negative_share", "mdna_firms"]].groupby(groups).mean().round(3)

## 7. Findings

*To fill in after the first full run on real data:*
* Coverage: how many filings and firm-years, and any gaps in the coverage table.
* How much of the Risk Factors text is repeated boilerplate.
* Does market-wide MD&A tone, or the negative share of new risk factors, move in the 60 days before episodes, or only during them (Section 6)? The formal test is in notebooks 14 and 15.
* Limits: 50 large survivors (firms that stayed in the index), quarterly data (a filing can be up to three months old), and FinBERT's 63% agreement in Salah's spot check.